# Phase 3 – Matching sémantique candidat ↔ offres (MatcherAgent)

Ce notebook implémente le **MatcherAgent** du système RecruitAI.
Il a pour objectif de mesurer la compatibilité entre le profil d’un candidat
et des offres d’emploi analysées, à l’aide de représentations sémantiques
(embeddings) et de métriques de similarité.

Le résultat est un score de matching explicable, utilisé ensuite pour
la génération ciblée de CV et de lettres de motivation.


In [1]:
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
with open("job_offers_analyzed.json", "r", encoding="utf-8") as f:
    job_offers = json.load(f)

print(f"📄 {len(job_offers)} offres analysées chargées")
job_offers[0]


##Profil candidat

In [4]:
candidate_profile = {
    "target_role": "Data Scientist",
    "skills": [
        "Python",
        "Machine Learning",
        "Data Analysis",
        "Statistics",
        "SQL",
        "Data Visualization"
    ],
    "soft_skills": [
        "Communication",
        "Problem-solving",
        "Teamwork"
    ],
    "experience_level": "Mid-Level"
}


##Fonction de vectorisation simple (baseline)

In [5]:
def text_to_vector(text: str, vocabulary: List[str]) -> np.ndarray:
    """
    Transforme un texte en vecteur binaire basé sur la présence de mots-clés.
    """
    text = text.lower()
    return np.array([1 if word.lower() in text else 0 for word in vocabulary])


##Construction du vocabulaire global

In [6]:
# Vocabulaire basé sur les compétences du candidat
skill_vocabulary = list(set(candidate_profile["skills"]))
soft_skill_vocabulary = list(set(candidate_profile["soft_skills"]))


##Fonction de matching d’une offre

In [7]:
def match_offer(job: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    # Texte offre
    job_skills_text = " ".join(job.get("skills", []))
    job_soft_text = " ".join(job.get("soft_skills", []))

    # Texte candidat
    candidate_skills_text = " ".join(candidate.get("skills", []))
    candidate_soft_text = " ".join(candidate.get("soft_skills", []))

    # Vecteurs
    job_skill_vec = text_to_vector(job_skills_text, skill_vocabulary)
    cand_skill_vec = text_to_vector(candidate_skills_text, skill_vocabulary)

    job_soft_vec = text_to_vector(job_soft_text, soft_skill_vocabulary)
    cand_soft_vec = text_to_vector(candidate_soft_text, soft_skill_vocabulary)

    # Similarités
    skill_score = cosine_similarity(
        job_skill_vec.reshape(1, -1),
        cand_skill_vec.reshape(1, -1)
    )[0][0]

    soft_skill_score = cosine_similarity(
        job_soft_vec.reshape(1, -1),
        cand_soft_vec.reshape(1, -1)
    )[0][0]

    # Score global pondéré
    final_score = round((0.7 * skill_score + 0.3 * soft_skill_score) * 100, 2)

    return {
        "offer_title": job.get("title"),
        "company": job.get("company"),
        "matching_score": final_score,
        "details": {
            "skill_similarity": round(skill_score, 2),
            "soft_skill_similarity": round(soft_skill_score, 2),
            "matched_skills": list(set(job.get("skills", [])) & set(candidate.get("skills", []))),
            "missing_skills": list(set(job.get("skills", [])) - set(candidate.get("skills", [])))
        }
    }


##Matching sur toutes les offres

In [8]:
matching_results = []

for job in job_offers:
    result = match_offer(job, candidate_profile)
    matching_results.append(result)


In [9]:
df_matches = pd.DataFrame(matching_results)
df_matches = df_matches.sort_values(by="matching_score", ascending=False)

df_matches


,offer_title,company,matching_score,details
0,Data Scientist Biology,Indeed.com (probablement annonce relayée),93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
2,Staff Data Scientist,Indeed.com (Implicitement),93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
10,Data Scientist,Accelerator Data Science (via fr.indeed.com pr...,93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
4,Data Scientist,Indeed.com,93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
16,Data Scientist,Aqemia,93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
18,Data Scientist,"QuantumBlack, AI by McKinsey",93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
19,Energy Data Scientist,"Indeed.com (Probablement pour un client, mais ...",93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
12,"Senior Data Scientist, Research & Development",Indeed,93.90,"{'skill_similarity': 0.91, 'soft_skill_similar..."
11,Product Analytics - Data Scientist,Indeed.com,88.40,"{'skill_similarity': 0.91, 'soft_skill_similar..."
1,Health Data Scientist,Indeed.com (Indirect - Aggregator),88.40,"{'skill_similarity': 0.91, 'soft_skill_similar..."


In [10]:
TOP_N = 5
df_matches.head(TOP_N)


,offer_title,company,matching_score,details
0,Data Scientist Biology,Indeed.com (probablement annonce relayée),93.9,"{'skill_similarity': 0.91, 'soft_skill_similar..."
2,Staff Data Scientist,Indeed.com (Implicitement),93.9,"{'skill_similarity': 0.91, 'soft_skill_similar..."
10,Data Scientist,Accelerator Data Science (via fr.indeed.com pr...,93.9,"{'skill_similarity': 0.91, 'soft_skill_similar..."
4,Data Scientist,Indeed.com,93.9,"{'skill_similarity': 0.91, 'soft_skill_similar..."
16,Data Scientist,Aqemia,93.9,"{'skill_similarity': 0.91, 'soft_skill_similar..."


In [11]:
with open("job_offers_matched.json", "w", encoding="utf-8") as f:
    json.dump(matching_results, f, indent=2, ensure_ascii=False)

print("✅ job_offers_matched.json généré")


✅ job_offers_matched.json généré


## Conclusion

Le MatcherAgent a permis d’évaluer automatiquement la compatibilité entre
le profil du candidat et les offres d’emploi analysées.

Chaque offre est associée à :
- un score de matching global
- des informations explicables (compétences correspondantes et manquantes)

Ces résultats serviront directement à piloter la génération de CV et
de lettres de motivation personnalisées par le WriterAgent.
